# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides guidance for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains multiple record sets, fields, and columns accessible by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Version:", getattr(metadata, 'version', 'N/A'))
print("License:", getattr(metadata, 'license', 'N/A'))


## 2. Data Overview
Review available record sets and the fields/columns within each, referencing them by their `@id`.

Entities in Croissant (datasets, record sets, fields, columns, etc.) are referenced by their `@id`.

Let's enumerate all record sets and their fields.

In [ ]:
# List all record sets in the dataset by their @id

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"Record Set name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List fields and columns for this record set
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {getattr(field, 'name', '')} (@id: {field.id}, Type: {getattr(field, 'data_type', '')})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {getattr(col, 'name', '')} (@id: {col.id}, Type: {getattr(col, 'data_type', '')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities must be referenced by `@id`.

> **Note:** Substitute the record set `@id` and field `@id`s displayed above. For this dataset, we will extract from the primary protein abundance table. If there are multiple record sets, we show how to load them all.

In [ ]:
# Map all available record set @id's for programmatic access
record_sets_ids = [rs.id for rs in record_sets]

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame columns for {record_set_id}: {df.columns.tolist()}")
    print(f"Number of records: {len(df)}\n")

# For this example, we pick the first record set:
primary_record_set_id = record_sets_ids[0] if record_sets_ids else None

if primary_record_set_id:
    print(f"Primary record set chosen: {primary_record_set_id}")
    display(dataframes[primary_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Now we can conduct typical processing steps. We need to identify numeric fields by their `@id` — locate relevant columns from the previous outputs (e.g. protein abundance, peptide counts, molecular weight).

In [ ]:
# Choose a numeric field, e.g., 'Molecular Weight' or 'Abundance' by its @id or column name

df = dataframes[primary_record_set_id]

# Try to pick a commonly found protein abundance/mass field
possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'if' or 'abundance' in col.lower() or 'weight' in col.lower() or 'count' in col.lower()]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.columns[0]  # fallback

print(f'Numeric field chosen for analysis: {numeric_field}')

# Filtering: For demonstration, threshold = 10
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Grouping by another attribute, if available
candidate_group_fields = [col for col in df.columns if col != numeric_field and df[col].nunique() < 10]
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    print(f\"Grouping by field: {group_field}\")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())
else:
    print("No suitable group field found.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset (e.g. distribution of abundance, correlation of peptide counts and molecular weight, etc.).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If there is a second numeric field, show scatterplot
other_numeric_fields = [col for col in df.columns if col != numeric_field and df[col].dtype.kind in 'if']
if other_numeric_fields:
    secondary_field = other_numeric_fields[0]
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[numeric_field], y=df[secondary_field])
    plt.xlabel(numeric_field)
    plt.ylabel(secondary_field)
    plt.title(f"{numeric_field} vs. {secondary_field}")
    plt.show()

## 6. Conclusion
In this notebook, you explored the FAIR^2 mass spectrometry protein data using the `mlcroissant` library. You learned how to enumerate available record sets, extract and process data using their `@id`, perform exploratory analysis, and visualize important features.

Possible next steps include more detailed analysis, integration with biological databases, and advanced machine learning workflows.
